In [8]:
from coupled_solver.utils.generate_mesh import _create_volumetric_mesh
_create_volumetric_mesh('sphere_6', algorithm=10)

sphere_6.xyzr file not detected.
Creating sphere_6.xyzr file
No se ha encontrado el archivo /home/fvicencio/non-linear_solver/molecules/sphere_6/sphere_6.xyzr, no se ha creado el archivo en formato .xyzr, si esta malla es de una esfera, ignorar este error.
Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Reading '/home/fvicencio/non-linear_solver/coupled_solver/boundary_mesh/sphere_6_ses.stl'...
Info    : Mesh is in binary format
Info    : 1324 facets in solid 0 
Info    : Done reading '/home/fvicencio/non-linear_solver/coupled_solver/boundary_mesh/sphere_6_ses.stl'
Info    : Reading '/home/fvicencio/non-linear_solver/coupled_solver/boundary_mesh/sphere_6_stern.stl'...
Info    : Mesh is in binary format
Info    : 1726 facets in solid 0 
Info    : Done reading '/home/fvicencio/non-linear_solver/coupled_solver/boundary_mesh/sphere_6_stern.stl'
Info    : Meshing 1D...
Info    : Done meshing 1D (Wall 1.63049e-05s, CPU 1.8e-05s)
Info    : Mes

In [1]:
import gmsh

gmsh.initialize()

# Cargar la malla existente
gmsh.open("coupled_solver/volumetric_mesh/sphere_8.msh")

# Guardarla directamente en formato VTK
gmsh.write("sphere_8.vtk")

gmsh.finalize()

Info    : Reading 'coupled_solver/volumetric_mesh/sphere_8.msh'...
Info    : 4 entities
Info    : 11075 nodes
Info    : 64390 elements
Info    : Done reading 'coupled_solver/volumetric_mesh/sphere_8.msh'
Info    : Writing 'sphere_8.vtk'...
Info    : Done writing 'sphere_8.vtk'


In [4]:
import gmsh

gmsh.initialize()
gmsh.open('coupled_solver/volumetric_mesh/1pgb.msh')
gmsh.write('1pgb.vtk')
gmsh.finalize()

Info    : Reading 'coupled_solver/volumetric_mesh/1pgb.msh'...
Info    : 4 entities
Info    : 83680 nodes
Info    : 547340 elements
Info    : Done reading 'coupled_solver/volumetric_mesh/1pgb.msh'                                                 
Info    : Writing '1pgb.vtk'...
Info    : Done writing '1pgb.vtk'


In [5]:
from dolfinx.io import gmshio
from mpi4py import MPI

mesh, cell_tags, facet_tags = gmshio.read_from_msh(
        'coupled_solver/volumetric_mesh/1pgb.msh', 
        MPI.COMM_WORLD, 
        rank=0, 
        gdim=3
        )

tdim = mesh.topology.dim
assert tdim == 3
fdim = tdim - 1

mesh.topology.create_connectivity(0, tdim)
mesh.topology.create_connectivity(tdim, 0)
mesh.topology.create_entities(fdim)
mesh.topology.create_connectivity(tdim, fdim)  # cell -> facet
mesh.topology.create_connectivity(fdim, tdim)  # facet -> cell
mesh.topology.create_connectivity(fdim, 0)     # facet -> vertex 
mesh.topology.create_entity_permutations()

Info    : Reading 'coupled_solver/volumetric_mesh/1pgb.msh'...
Info    : 4 entities
Info    : 83680 nodes
Info    : 547340 elements
Info    : Done reading 'coupled_solver/volumetric_mesh/1pgb.msh'                                                 


In [6]:
solute_cells = cell_tags.find(1)
stern_cells = cell_tags.find(2)

In [9]:
print(solute_cells.size)
print(stern_cells.size)

147833
347923


In [7]:
from dolfinx import fem
fenics_space = fem.functionspace(mesh, ('Lagrange', 2))

In [10]:
solute_dofs = set()
for cell_idx in solute_cells:
    cell_ndofs = fenics_space.dofmap.cell_dofs(cell_idx)
    for dof in cell_ndofs:
        solute_dofs.add(dof)

In [11]:
solute_dofs = list(solute_dofs)

In [12]:
x_dofs = fenics_space.tabulate_dof_coordinates()

In [13]:
x_dofs_solute = x_dofs[solute_dofs]

In [3]:
import numpy as np

arreglo = np.zeros(10)
index = [1, 3, 5]
valores = [1, 2, 3]

arreglo[index] = valores
print(arreglo)

[0. 1. 0. 2. 0. 3. 0. 0. 0. 0.]


In [16]:
x_dofs.shape

(669619, 3)